# copy_paste augmentation パイロット — fold0、検出器側レバー(v57の代替機構)

v57 で密集サンプル複製(dense2x)は dense FN を減らしたが FP 爆増で不採用。
シーン複製の代わりに **インスタンス貼付(ultralytics copy_paste)** で密集
シーンのリコールを上げられるか検証する。

| 構成 | 内容 |
|---|---|
| `baseline` | V30完全同一(copy_paste=0)。旧 baseline の自動再利用なし |
| `copypaste` | copy_paste=0.2 のみ追加(他のハイパラはV30不変) |

学習レシピはV30と完全同一(yolo11m-seg/imgsz1280/70epoch/AdamW lr0=7e-4)。
fold0基準: refined PQ 0.4393 / raw YOLO PQ 0.3977。
GPU必須、A100推奨。manifestはDriveへ逐次追記(中断再開可)。


In [ ]:
# [1/8] Drive接続、設定、前提の確認
!pip -q install ultralytics pycocotools
import hashlib, json, math, random, shutil, sys, time, warnings
from datetime import datetime, timezone
from importlib.metadata import version
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

class CFG:
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v60_copypaste")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    YOLO_ROOT = LOCAL_ROOT / "yolo_datasets"
    RUNS_ROOT = DRIVE_ROOT / "runs" / "v60_copypaste"
    FOLD = 0
    PROTOCOL_SHA256 = '675185adccabb237de7499e1b4971445bb10051c32eb0335650921e2c9f3b071'
    REUSE_VERIFIED_BASELINE = False
    # --- build_v30_prize_colab.py と完全同一。触らないこと ---
    YOLO_MODEL = "yolo11m-seg.pt"
    YOLO_IMGSZ = 1280
    YOLO_BATCH = 4
    YOLO_EPOCHS = 70
    YOLO_PATIENCE = 14
    YOLO_MASK_RATIO = 2
    # --- OOF評価は確定構成(build_v56/v57系と同一。refinerは既存ckptを使い回す) ---
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    CONF = 0.26
    PREDICT_CONF_FLOOR = 0.08
    PREDICT_IOU = 0.55
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    BASELINE_PQ_REFINED = 0.4393
    DENSE_QUANTILE = 0.75
    COPY_PASTE = 0.2
    SEEDS = (3407, 8407)
    CONFIGS = ('baseline', 'copypaste')
    VERIFIED_BASELINE_MANIFEST = DRIVE_ROOT / "artifacts" / "v60_verified_baseline_fold0_manifest.json"
    MANIFEST = DRIVE_ROOT / "artifacts" / "v60_copypaste_fold0_manifest.json"

for path in [CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.YOLO_ROOT, CFG.RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

required = {
    "competition data": CFG.DATA_CACHE,
    "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
    "前処理画像キャッシュ": CFG.PREPROC_CACHE,
    f"fold{CFG.FOLD} 標準refiner": CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required})
print({"GPU": torch.cuda.get_device_name(0)})


In [ ]:
# [2/8] COCOとfoldを読み、密度スライス用のcutを計算し、前処理画像をローカルSSDへ展開
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"学習annotation JSONが見つかりません: {base}")
    return matches[0].parents[2]

TRAIN_JSON = next(find_dataset_root(CFG.DATA_CACHE).rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
with open(TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)

images = {item["id"]: item for item in coco["images"]}
annotations_by_image = defaultdict(list)
for ann in coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)

def annotation_rank(image_id):
    anns = annotations_by_image[image_id]
    return len(anns), sum(float(a.get("area", 0)) for a in anns)

file_to_reference_id = {
    name: max(ids, key=annotation_rank) for name, ids in file_to_ids.items()
}
train_files = sorted(file_to_ids)

fold_frame = pd.read_csv(CFG.ARTIFACT_ROOT / "grouped_folds.csv")
if fold_frame["file_name"].duplicated().any() or not fold_frame["fold"].isin([0, 1, 2]).all():
    raise ValueError("Invalid or duplicate fold assignments")
fold_of_file = dict(zip(fold_frame["file_name"], fold_frame["fold"].astype(int)))
if set(train_files) != set(fold_of_file):
    raise ValueError("Fold assignments do not cover exactly the consumed files")
train_files_fold = [n for n in train_files if fold_of_file[n] != CFG.FOLD]
valid_files_fold = [n for n in train_files if fold_of_file[n] == CFG.FOLD]
if not train_files_fold or not valid_files_fold:
    raise ValueError("Both training and validation partitions must be nonempty")
assert not (set(train_files_fold) & set(valid_files_fold)), "fold分割が壊れている"

def noncrowd_count(image_id):
    return sum(1 for a in annotations_by_image[image_id] if not a.get("iscrowd", 0))

# v57と同じ定義: cutは学習サンプル(file,annotator)のGT数q75(dense/sparseスライス用)
train_sample_counts = [
    noncrowd_count(image_id)
    for name in train_files_fold
    for image_id in file_to_ids[name]
]
DENSE_CUT = float(np.quantile(train_sample_counts, CFG.DENSE_QUANTILE))
val_density = {
    name: float(np.mean([noncrowd_count(i) for i in file_to_ids[name]]))
    for name in valid_files_fold
}

for name in tqdm(train_files_fold + valid_files_fold, desc="前処理画像をSSDへ"):
    local = CFG.PREPROC_DIR / name
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CFG.PREPROC_CACHE / name, local)

print({"観測": len(train_files), "除外train": len(train_files_fold),
       "検証(OOF)": len(valid_files_fold),
       "dense_cut": DENSE_CUT,
       "dense検証画像数": int(sum(v >= DENSE_CUT for v in val_density.values()))})
def atomic_save(path, value, is_checkpoint=False):
    import os
    import tempfile
    path = Path(path)
    payload = None if is_checkpoint else json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode('utf-8')
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='wb', dir=path.parent, suffix='.tmp', delete=False) as handle:
            temporary = Path(handle.name)
            if is_checkpoint:
                torch.save(value, handle)
            else:
                handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def require_input_contract(path, expected):
    if not Path(path).is_file() or json.loads(Path(path).read_text(encoding="utf-8")) != expected:
        raise ValueError("Input contract differs or is absent; preserve artifacts and choose a fresh path")

def runtime_measurement(scope, started):
    try:
        peak, peak_note = int(torch.cuda.max_memory_allocated()), None
    except Exception as error:
        peak, peak_note = None, f"CUDA statistics unavailable: {type(error).__name__}"
    return {"service": "Google Colab", "project": str(CFG.DRIVE_ROOT),
            "notebook": f"solar_filament_v60_copypaste_fold{CFG.FOLD}_colab.ipynb",
            "gpu_name": torch.cuda.get_device_name(0), "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "measurement_scope": scope, "duration_seconds": round(time.time() - started, 3),
            "peak_vram_bytes": peak, "peak_vram_unavailable_reason": peak_note,
            "compute_units_used": None, "compute_units_note": "Unavailable from runtime API; record from Colab UI."}

def validate_run_row(row):
    if (not isinstance(row, dict) or row.get("config") not in CFG.CONFIGS or row.get("seed") not in CFG.SEEDS
            or row.get("fold") != CFG.FOLD or row.get("input_contract") != PILOT_CONTRACT
            or row.get("copy_paste") != (CFG.COPY_PASTE if row["config"] == "copypaste" else 0.0)):
        raise ValueError("Run arm/seed/fold/input contract differs")
    for mode in ("raw", "refined"):
        metrics = row.get(mode)
        if not isinstance(metrics, dict):
            raise ValueError("Missing quality metrics")
        for name in ("pq", "sq", "rq"):
            value = metrics.get(name)
            if type(value) not in (int, float) or not math.isfinite(value) or not 0 <= value <= 1:
                raise ValueError("Invalid quality metric")
        for name in ("tp", "fp", "fn"):
            if type(metrics.get(name)) is not int or metrics[name] < 0:
                raise ValueError("Invalid match count")
    for name in ("fn_dense_val", "fn_sparse_val"):
        if type(row.get(name)) is not int or row[name] < 0:
            raise ValueError("Invalid density count")
    checkpoint = Path(row.get("checkpoint", ""))
    if not checkpoint.is_file() or row.get("checkpoint_sha256") != file_sha256(checkpoint):
        raise ValueError("Checkpoint bytes differ or are absent")
    return row

def read_pilot_manifest():
    if not CFG.MANIFEST.exists():
        return {"experiment_id": f"v60_copypaste_fold{CFG.FOLD}", "input_contract": PILOT_CONTRACT, "runs": []}
    manifest = json.loads(CFG.MANIFEST.read_text(encoding="utf-8"))
    if (not isinstance(manifest, dict) or manifest.get("experiment_id") != f"v60_copypaste_fold{CFG.FOLD}"
            or manifest.get("input_contract") != PILOT_CONTRACT or not isinstance(manifest.get("runs"), list)):
        raise ValueError("Manifest input contract differs or is absent; retain historical files")
    seen = set()
    for row in manifest["runs"]:
        validate_run_row(row)
        key = (row["config"], row["seed"])
        if key in seen:
            raise ValueError("Duplicate pilot run")
        seen.add(key)
    return manifest

def reuse_baselines(source):
    if not CFG.REUSE_VERIFIED_BASELINE:
        return []
    if not isinstance(source, dict) or source.get("input_contract") != PILOT_CONTRACT:
        raise ValueError("Baseline equivalence is unverified; rerun paired baseline")
    result, seen = [], set()
    for row in source.get("runs", []):
        if row.get("config") != "baseline":
            continue
        validate_run_row(row)
        key = (row["config"], row["seed"])
        if key in seen:
            raise ValueError("Duplicate reused baseline")
        seen.add(key)
        result.append({**row, "reused_from": "explicit_contract_checked_baseline"})
    return result

from ultralytics import YOLO
initial_yolo = YOLO(CFG.YOLO_MODEL)
INITIAL_YOLO_PATH = Path(initial_yolo.ckpt_path).resolve(strict=True)
if initial_yolo.task != "segment":
    raise ValueError("Copy-paste pilot requires the segmentation pretrained model")
del initial_yolo
PILOT_CONTRACT = {
    "schema_version": 1, "fold": CFG.FOLD, "protocol_sha256": CFG.PROTOCOL_SHA256,
    "annotations_sha256": file_sha256(TRAIN_JSON),
    "folds_sha256": file_sha256(CFG.ARTIFACT_ROOT / "grouped_folds.csv"),
    "initial_yolo_sha256": file_sha256(INITIAL_YOLO_PATH),
    "refiner_sha256": file_sha256(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt"),
    "preprocessed_sha256": hashlib.sha256(json.dumps({name: file_sha256(CFG.PREPROC_DIR / name)
        for name in sorted(train_files_fold + valid_files_fold)}, sort_keys=True).encode()).hexdigest(),
    "recipe": {name: getattr(CFG, name) for name in ("YOLO_MODEL", "YOLO_EPOCHS", "YOLO_IMGSZ", "YOLO_BATCH",
        "YOLO_PATIENCE", "YOLO_MASK_RATIO", "CROP_SIZE", "CROP_CONTEXT", "CONF", "PREDICT_CONF_FLOOR",
        "PREDICT_IOU", "REFINER_THRESHOLD", "DROP_THRESHOLD", "MIN_AREA", "DENSE_QUANTILE", "COPY_PASTE")},
    "seeds": list(CFG.SEEDS), "configs": list(CFG.CONFIGS),
    "runtime_versions": {"python": sys.version, "torch": torch.__version__, "numpy": np.__version__,
        "opencv": cv2.__version__, "ultralytics": version("ultralytics"), "pycocotools": version("pycocotools")},
}
PILOT_CONTRACT = json.loads(json.dumps(PILOT_CONTRACT, sort_keys=True, allow_nan=False))
read_pilot_manifest()  # No legacy/mixed results may reach GPU evaluation.


In [ ]:
# [3/8] YOLOデータセット構築(V30と完全同一 — copy_pasteはデータセットではなく学習kwargなので1構成のみ)
def polygon_line(ann, width, height):
    polygons = [p for p in ann.get("segmentation", []) if len(p) >= 6]
    if not polygons:
        return None
    # MAGFiLO specifies one polygon per filament. Largest is a safe fallback for
    # malformed/multipart records without turning one filament into many labels.
    polygon = max(polygons, key=len)
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    points[:, 0] = np.clip(points[:, 0] / width, 0, 1)
    points[:, 1] = np.clip(points[:, 1] / height, 0, 1)
    return "0 " + " ".join(f"{value:.6f}" for value in points.reshape(-1))

def link_or_copy(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return
    try:
        destination.symlink_to(source)
    except OSError:
        shutil.copy2(source, destination)

def write_variant(fold_root, split, file_name, image_id, suffix):
    info = images[image_id]
    stem = Path(file_name).stem + suffix
    image_out = fold_root / "images" / split / f"{stem}.jpeg"
    label_out = fold_root / "labels" / split / f"{stem}.txt"
    link_or_copy((CFG.PREPROC_DIR / file_name).resolve(), image_out)
    lines = []
    for ann in annotations_by_image[image_id]:
        if ann.get("iscrowd", 0):
            continue
        line = polygon_line(ann, info["width"], info["height"])
        if line:
            lines.append(line)
    label_out.parent.mkdir(parents=True, exist_ok=True)
    label_out.write_text("\n".join(lines), encoding="utf-8")

def build_fold_dataset():
    # データセットはV30と完全同一の1種のみ(baseline/copypaste共用)
    root = CFG.YOLO_ROOT / "shared" / f"fold_{CFG.FOLD}"
    contract_path = root / "INPUT_CONTRACT.json"
    if (root / "data.yaml").exists():
        require_input_contract(contract_path, PILOT_CONTRACT)
        print(f"既存データセットを再利用 {root}")
        return root / "data.yaml"
    if root.exists() and any(root.iterdir()):
        raise ValueError("Incomplete dataset: preserve it and choose a fresh pilot path")
    for name in tqdm(train_files_fold, desc=f"build fold{CFG.FOLD} train"):
        for image_id in file_to_ids[name]:
            write_variant(root, "train", name, image_id, f"__labeler_{image_id}")
    for file_name in tqdm(valid_files_fold, desc=f"build fold{CFG.FOLD} val"):
        image_id = file_to_reference_id[file_name]
        write_variant(root, "val", file_name, image_id, "")
    yaml_path = root / "data.yaml"
    yaml_path.write_text(
        f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
        encoding="utf-8",
    )
    atomic_save(contract_path, PILOT_CONTRACT)
    return yaml_path


In [ ]:
# [4/8] 学習(V30と同一ハイパラ、copypaste構成のみcopy_paste追加) — runごとに再開対応
from ultralytics import YOLO

def training_kwargs(config, seed, yaml_path, run_name):
    return dict(
            data=str(yaml_path),
            epochs=CFG.YOLO_EPOCHS,
            imgsz=CFG.YOLO_IMGSZ,
            batch=CFG.YOLO_BATCH,
            device=0,
            workers=4,
            patience=CFG.YOLO_PATIENCE,
            seed=seed,
            deterministic=True,
            single_cls=True,
            mask_ratio=CFG.YOLO_MASK_RATIO,
            overlap_mask=False,
            amp=True,
            cache="disk",
            optimizer="AdamW",
            lr0=7e-4,
            lrf=0.05,
            weight_decay=5e-4,
            warmup_epochs=3,
            degrees=180,
            translate=0.04,
            scale=0.25,
            fliplr=0.5,
            flipud=0.5,
            hsv_h=0.0,
            hsv_s=0.0,
            hsv_v=0.18,
            mosaic=0.20,
            close_mosaic=10,
            project=str(CFG.RUNS_ROOT),
            name=run_name,
            exist_ok=True,
            plots=True,
            save=True,
            save_period=5,
            verbose=True,
            copy_paste=CFG.COPY_PASTE if config == "copypaste" else 0.0,
        )

def train_yolo(config, seed):
    run_name = f"yolo_v60_{config}_s{seed}_fold{CFG.FOLD}"
    run_dir = CFG.RUNS_ROOT / run_name
    best_path = run_dir / "weights" / "best.pt"
    last_path = run_dir / "weights" / "last.pt"
    done_path = run_dir / "TRAINING_COMPLETE"
    if config not in CFG.CONFIGS or seed not in CFG.SEEDS:
        raise ValueError("Unknown pilot arm or seed")
    contract = {**PILOT_CONTRACT, "config": config, "seed": seed}
    contract_path = run_dir / "INPUT_CONTRACT.json"
    if contract_path.exists():
        require_input_contract(contract_path, contract)
    elif best_path.exists() or last_path.exists() or done_path.exists():
        raise ValueError("Checkpoint has no input contract; retain it and choose a fresh path")
    else:
        run_dir.mkdir(parents=True, exist_ok=True)
        atomic_save(contract_path, contract)
    if done_path.exists():
        completion = json.loads(done_path.read_text(encoding="utf-8"))
        if (completion.get("input_contract") != contract or not best_path.is_file()
                or completion.get("checkpoint_sha256") != file_sha256(best_path)):
            raise ValueError("Completed checkpoint contract/bytes differ; retain artifacts")
        return best_path
    if file_sha256(INITIAL_YOLO_PATH) != PILOT_CONTRACT["initial_yolo_sha256"]:
        raise ValueError("Initial YOLO bytes changed after contract construction")

    yaml_path = build_fold_dataset()
    seed_everything(seed)
    if last_path.exists():
        print(f"{run_name}: resuming {last_path}")
        model = YOLO(str(last_path))
        model.train(resume=True)
    else:
        model = YOLO(str(INITIAL_YOLO_PATH))
        model.train(**training_kwargs(config, seed, yaml_path, run_name))
    if not best_path.exists():
        raise FileNotFoundError(f"Training ended without {best_path}")
    atomic_save(done_path, {"input_contract": contract, "checkpoint_sha256": file_sha256(best_path)})
    return best_path


In [ ]:
# [5/8] MediSegUNet (solar_filament/models/mediseg_models.py と同一。標準refinerの評価に使用)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


In [ ]:
# [6/8] 推論・後処理・採点関数の定義(build_v57と同一経路: raw + refined + 密度スライスFN)
from pycocotools import mask as mask_utils

if file_sha256(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt") != PILOT_CONTRACT["refiner_sha256"]:
    raise ValueError("Fixed refiner bytes changed")
refiner = MediSegUNet(3, 1, False).cuda().eval()
_ckpt = torch.load(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
                    map_location="cuda", weights_only=False)
refiner.load_state_dict(_ckpt.get("best_model") or _ckpt["model"], strict=True)
del _ckpt

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return output, confidences

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposal(image, proposal):
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= CFG.REFINER_THRESHOLD
    return refined & proposal  # shrink(確定構成)

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

class PanopticQuality:
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.tp_ious = []
        self.fp_count = 0
        self.fn_count = 0
    def add_matrix(self, matrix, n_gt, n_pred):
        if n_gt == 0:
            self.fp_count += n_pred; return
        if n_pred == 0:
            self.fn_count += n_gt; return
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        self.tp_ious.extend(scores)
        self.fp_count += int((hits.sum(axis=0) == 0).sum())
        self.fn_count += int((hits.sum(axis=1) == 0).sum())
    @property
    def denominator(self):
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count
    @property
    def score(self):
        d = self.denominator
        return float(sum(self.tp_ious) / d) if d > 0 else 0.0
    def sq_rq(self):
        tp = len(self.tp_ious)
        sq = float(sum(self.tp_ious) / tp) if tp > 0 else 0.0
        rq = float(tp / (tp + 0.5 * self.fp_count + 0.5 * self.fn_count)) if (tp + self.fp_count + self.fn_count) > 0 else 0.0
        return sq, rq
    def summary(self):
        sq, rq = self.sq_rq()
        return {"pq": self.score, "sq": sq, "rq": rq, "tp": len(self.tp_ious),
                "fp": self.fp_count, "fn": self.fn_count, "denominator": self.denominator}

gt_rles = {}
for image_id, info in images.items():
    height, width = int(info["height"]), int(info["width"])
    gt_rles[image_id] = [to_rle(ann, height, width) for ann in annotations_by_image[image_id] if not ann.get("iscrowd", 0)]

def evaluate_checkpoint(best_path, config, seed):
    """raw(生YOLO)とrefined(確定構成)の両方でPQ採点 + 密度スライス別FNを記録。"""
    yolo_model = YOLO(str(best_path))
    meters = {"raw": PanopticQuality(), "refined": PanopticQuality()}
    fn_dense = fn_sparse = 0
    for name in tqdm(valid_files_fold, desc=f"{config}/s{seed} fold{CFG.FOLD} OOF採点"):
        image_path = CFG.PREPROC_DIR / name
        result = yolo_model.predict(str(image_path), imgsz=CFG.YOLO_IMGSZ,
                                     conf=CFG.PREDICT_CONF_FLOOR, iou=CFG.PREDICT_IOU,
                                     agnostic_nms=True, retina_masks=True, verbose=False)[0]
        masks, confidences = result_masks(result)
        keep = [i for i, (m, s) in enumerate(zip(masks, confidences)) if s >= CFG.CONF and m.sum() >= CFG.MIN_AREA]
        proposals = [masks[i] for i in keep]
        scores = confidences[keep] if len(keep) else np.zeros(0, np.float32)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        refined = [refine_proposal(image, m) for m in proposals]
        predictions = {"raw": apply_ownership(proposals, scores),
                       "refined": apply_ownership(refined, scores)}
        pred_rles_by_mode = {}
        for mode, preds in predictions.items():
            pred_rles = []
            if preds:
                encoded = mask_utils.encode(np.asfortranarray(np.stack(preds, axis=-1).astype(np.uint8)))
                pred_rles = encoded if isinstance(encoded, list) else [encoded]
            pred_rles_by_mode[mode] = pred_rles
        file_fn = 0
        for image_id in file_to_ids[name]:
            truth = gt_rles[image_id]
            for mode in ("raw", "refined"):
                matrix = iou_matrix(truth, pred_rles_by_mode[mode])
                meters[mode].add_matrix(matrix, len(truth), len(pred_rles_by_mode[mode]))
                if mode == "refined":
                    # FN(PQのIoU>0.5判定と同じ)をGT側で数え、annotator×imageで加算
                    if matrix.size:
                        file_fn += int((matrix.max(axis=1) <= 0.5).sum())
                    else:
                        file_fn += len(truth)
        if val_density[name] >= DENSE_CUT:
            fn_dense += file_fn
        else:
            fn_sparse += file_fn
    del yolo_model
    torch.cuda.empty_cache()
    return {"raw": meters["raw"].summary(), "refined": meters["refined"].summary(),
            "fn_dense_val": fn_dense, "fn_sparse_val": fn_sparse,
            "n_dense_val_files": int(sum(v >= DENSE_CUT for v in val_density.values()))}


In [ ]:
# [7/8] runを直列実行し、runごとにmanifestをDriveへ逐次追記(途中再開対応)
manifest = read_pilot_manifest()
rows = manifest["runs"]
done_keys = {(r["config"], r["seed"]) for r in rows}
if CFG.REUSE_VERIFIED_BASELINE and CFG.VERIFIED_BASELINE_MANIFEST.exists():
    for row in reuse_baselines(json.loads(CFG.VERIFIED_BASELINE_MANIFEST.read_text(encoding="utf-8"))):
        if (row["config"], row["seed"]) not in done_keys:
            rows.append(row)
            done_keys.add((row["config"], row["seed"]))
manifest.pop("failure", None)
manifest["status"] = "in_progress"
started = time.time()
scope = "pilot_preflight"
try:
    for config in CFG.CONFIGS:
        for seed in CFG.SEEDS:
            if (config, seed) in done_keys:
                continue
            run_dir = CFG.RUNS_ROOT / f"yolo_v60_{config}_s{seed}_fold{CFG.FOLD}"
            scope = "evaluation_only" if (run_dir / "TRAINING_COMPLETE").exists() else (
                "resumed_training_and_evaluation" if (run_dir / "weights/last.pt").exists() else "training_and_evaluation")
            started = time.time()
            torch.cuda.reset_peak_memory_stats()
            best_path = train_yolo(config, seed)
            checkpoint_sha = file_sha256(best_path)
            metrics = evaluate_checkpoint(best_path, config, seed)
            if file_sha256(best_path) != checkpoint_sha:
                raise ValueError("Evaluation checkpoint bytes changed during evaluation")
            row = {"config": config, "seed": seed, "fold": CFG.FOLD,
                   "checkpoint": str(best_path), "checkpoint_sha256": checkpoint_sha,
                   "input_contract": PILOT_CONTRACT,
                   "copy_paste": CFG.COPY_PASTE if config == "copypaste" else 0.0,
                   "execution": runtime_measurement(scope, started), **metrics}
            validate_run_row(row)
            rows.append(row)
            done_keys.add((config, seed))
            manifest["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
            atomic_save(CFG.MANIFEST, manifest)
            print(json.dumps(row, ensure_ascii=False, allow_nan=False))
except (Exception, KeyboardInterrupt) as error:
    manifest["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
    manifest["failure"] = {"error_type": type(error).__name__, **runtime_measurement(scope, started)}
    atomic_save(CFG.MANIFEST, manifest)
    raise
manifest["status"] = "complete"
atomic_save(CFG.MANIFEST, manifest)

# --- 集計と採否の目安 ---
print("\n=== 集計 ===")
for row in rows:
    delta_ref = row["refined"]["pq"] - CFG.BASELINE_PQ_REFINED
    print(f"{row['config']}/s{row['seed']}: raw PQ {row['raw']['pq']:.4f} | "
          f"refined PQ {row['refined']['pq']:.4f} (Δ{delta_ref:+.4f} vs fold{CFG.FOLD}基準) | "
          f"FN dense/sparse {row['fn_dense_val']}/{row['fn_sparse_val']}")

pair = {}
for row in rows:
    pair.setdefault(row["seed"], {})[row["config"]] = row
print("\n=== seed別 paired Δ(copypaste - baseline) ===")
for seed, arms in sorted(pair.items()):
    if "baseline" in arms and "copypaste" in arms:
        d_raw = arms["copypaste"]["raw"]["pq"] - arms["baseline"]["raw"]["pq"]
        d_ref = arms["copypaste"]["refined"]["pq"] - arms["baseline"]["refined"]["pq"]
        d_fn = arms["copypaste"]["fn_dense_val"] - arms["baseline"]["fn_dense_val"]
        d_fp = arms["copypaste"]["refined"]["fp"] - arms["baseline"]["refined"]["fp"]
        print(f"seed {seed}: Δraw {d_raw:+.4f} | Δrefined {d_ref:+.4f} | "
              f"ΔFN(dense) {d_fn:+d} | ΔFP {d_fp:+d}")
print("\n採否の目安: 2seedで符号一致かつdense FNが減ればfold1/2で独立確認。"
      "v57同様「FN改善があってもFP爆増でPQが下がる」場合は不採用(conf選択・独立確認は別途必要)。"
      "copypasteでもdense FNが減らなければ、検出器側の密集リコール系レバーは総括して不採用方向。")
